In [1]:
from z3 import *

Слегка расширенный пример из 1.2

In [11]:
x, y, z = Ints('x y z')
g = Goal()
g.add(x>0, y>0, z<0, x==y+1+1+z)

t1 = Tactic("simplify")
t2 = Tactic("solve-eqs")

t = Then(t1, t2)
print(t(g))

[[Not(x <= 0), Not(x + -1*z <= 2), Not(0 <= z)]]


Отсюда следует простая стратегия для линейных уравнений

In [ ]:
linear = Then('simplify', 'solve-eqs', 'smt').solver()

Невыполнимая цель для тактики

In [8]:
g = Goal(); g.add(x > 5, x < 3)
print(Tactic('simplify')(g))

[[Not(x <= 5), Not(3 <= x)]]


Пример со split-clause

In [9]:
g = Goal(); g.add(Or(x==1, x==2), Or(y==1, y==2), x+y==4)
r = Repeat(OrElse('split-clause','skip'))(g)

Выбор стратегии - тут я конечно сильно засел. Основной вопрос - как именно определить, к какому классу относится задача. Тут пришлось воспользоваться помощью клода, получилось довольно интересн - делаем отдельный классификатор, и берем специализированную стратегию, дополнительно узнал за метрики

In [10]:
def klass(g):
    if Probe('is-qfbv')(g) > 0:                                    return 'BV'
    if Probe('is-qfnra')(g) > 0:                                   return 'NRA'
    if Probe('is-qflia')(g) > 0 and Probe('is-unbounded')(g) == 0:  return 'LIA-bounded'
    if Probe('is-qflia')(g) > 0:                                   return 'LIA'
    return 'other'

STRATEGIES = {
 'BV':          Then('simplify', 'solve-eqs', 'bit-blast', 'sat'),
 'NRA':         Tactic('qfnra-nlsat'),
 'LIA-bounded': Then(With('simplify', arith_lhs=True, som=True),
                     'normalize-bounds', 'lia2pb', 'pb2bv', 'bit-blast', 'sat'),
 'LIA':         Then('simplify', 'propagate-values', 'solve-eqs', 'smt'),
 'other':       Tactic('smt'),
}

def solve_smart(title, *cs):
    g = Goal(); g.add(*cs)
    k = klass(g)                      # 1. определили класс
    s = STRATEGIES[k].solver()        # 2. взяли стратегию с полки
    s.add(*cs)                        # 3. решили
    print(title, "| класс:", k, "|", s.check(), s.model())

A) p|q==13, p>q                    класс: BV           sat  [q = 0, p = 13]
B) u²+v²==1, u·v>¼                 класс: NRA          sat  [u = -1/2, v = -0.866...]
C) 0<x<10, 0<y<10, 2x+3y==17       класс: LIA-bounded  sat  [x = 1, y = 5]
D) x>0, y>0, x==y+2                класс: LIA          sat  [x = 3, y = 1]
E) ForAll x: x*x >= 0              класс: other        sat  []